In [3]:
import torch

In [4]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [13]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [15]:
din = 3
dout = 2

In [10]:
context_length = inputs[0].size()[0]
one_mat = torch.ones(context_length, context_length)

In [40]:
self_attended = SelfAttention_v2(din, dout)
queries = self_attended.W_query(inputs)
keys = self_attended.W_key(inputs)
values = self_attended.W_value(inputs)
attention_scores = queries @ keys.T
attention_weights = torch.softmax(attention_scores / keys.shape[0] ** 0.5, dim = -1)
print(attention_weights)

tensor([[0.1668, 0.1663, 0.1664, 0.1665, 0.1684, 0.1656],
        [0.1671, 0.1653, 0.1656, 0.1663, 0.1727, 0.1631],
        [0.1671, 0.1653, 0.1656, 0.1662, 0.1728, 0.1629],
        [0.1668, 0.1659, 0.1661, 0.1666, 0.1698, 0.1649],
        [0.1679, 0.1654, 0.1658, 0.1655, 0.1735, 0.1618],
        [0.1663, 0.1660, 0.1661, 0.1670, 0.1687, 0.1659]],
       grad_fn=<SoftmaxBackward0>)


In [30]:
context_length = attention_scores.shape[0]
simple_mask = torch.tril(torch.ones(context_length, context_length))
print(simple_mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [33]:
masked_attention_weights = attention_weights * simple_mask
masked_attention_weights = masked_attention_weights / masked_attention_weights.sum(dim = 1)
print(masked_attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0006, 0.5065, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0001, 0.5063, 0.3348, 0.0000, 0.0000, 0.0000],
        [0.9925, 0.4965, 0.3292, 0.2437, 0.0000, 0.0000],
        [0.9866, 0.4970, 0.3296, 0.2440, 0.1878, 0.0000],
        [0.9978, 0.4990, 0.3306, 0.2427, 0.1843, 0.1714]],
       grad_fn=<DivBackward0>)


In [60]:
# Now make it properly causal
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_attention_scores  = attention_scores.masked_fill(mask.bool(), -torch.inf)
masked_attention_weights = torch.softmax(masked_attention_scores / keys.shape[-1] ** 0.5 , dim = 1)
dropout = nn.Dropout(0.5)
masked_attention_weights = dropout(masked_attention_weights)
print(masked_attention_weights)
context_vector = masked_attention_weights @ values
print(context_vector)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.9906, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6616, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5023, 0.4978, 0.0000, 0.5013, 0.0000, 0.0000],
        [0.4012, 0.3908, 0.3923, 0.3911, 0.0000, 0.0000],
        [0.0000, 0.3310, 0.3314, 0.0000, 0.3403, 0.3308]],
       grad_fn=<MulBackward0>)
tensor([[-0.7831,  1.1316],
        [-0.0300,  0.0923],
        [-0.0200,  0.0617],
        [-0.1696,  0.3002],
        [-0.1535,  0.2803],
        [-0.0518,  0.1032]], grad_fn=<MmBackward0>)


In [75]:
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)

torch.Size([2, 6, 3])


In [79]:
# Causal Attention Class
class CausalAttention(nn.Module):
  def __init__(self, din, dout, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.Wq = nn.Linear(din, dout, bias=qkv_bias)
    self.Wk = nn.Linear(din, dout, bias=qkv_bias)
    self.Wv = nn.Linear(din, dout, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

  def forward(self, x):
    b, num_tokens, din = x.shape
    queries = self.Wq(x)
    keys = self.Wk(x)
    values = self.Wv(x)
    attention_scores = queries @ keys.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(self.mask.bool(), -torch.inf)
    attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim = -1)
    attention_weights = self.dropout(attention_weights)
    context_vector = attention_weights @ values
    return context_vector



In [82]:
cat = CausalAttention(din, dout, context_length, 0.05)
context_vector = cat.forward(batch)
print(context_vector)

tensor([[[0.3248, 0.0071],
         [0.4400, 0.0356],
         [0.4700, 0.0421],
         [0.4459, 0.0478],
         [0.2293, 0.0006],
         [0.3893, 0.0368]],

        [[0.3248, 0.0071],
         [0.4400, 0.0356],
         [0.4700, 0.0421],
         [0.4459, 0.0478],
         [0.3345, 0.0115],
         [0.3893, 0.0368]]], grad_fn=<UnsafeViewBackward0>)
